In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimization Solver: Uma Qiskit Function do Q-CTRL Fire Opal
*Consulte a [referência da API](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)*

> **Note:** As Qiskit Functions são um recurso experimental disponível apenas para usuários dos planos IBM Quantum&reg; Premium, Flex e On-Prem (via IBM Quantum Platform API). Elas estão em status de lançamento de pré-visualização e sujeitas a alterações.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## Visão geral
Com o Fire Opal Optimization Solver, você pode resolver problemas de otimização em escala utilitária em hardware quântico sem precisar de conhecimento especializado em computação quântica. Basta inserir a definição do problema em alto nível e o Solver cuida do resto. Todo o fluxo de trabalho é ciente de ruído e utiliza o [Gerenciamento de Desempenho do Fire Opal](/guides/q-ctrl-performance-management) por baixo dos panos. O Solver entrega consistentemente soluções precisas para problemas classicamente desafiadores, mesmo em escala de dispositivo completo nos maiores QPUs da IBM&reg;.

O Solver é flexível e pode ser usado para resolver problemas de otimização combinatória definidos como funções objetivo ou grafos arbitrários. Os problemas não precisam ser mapeados para a topologia do dispositivo. Tanto problemas sem restrições quanto com restrições são resolúveis, desde que as restrições possam ser formuladas como termos de penalidade. Os exemplos incluídos neste guia demonstram como resolver um problema de otimização em escala utilitária sem restrições e um com restrições, usando diferentes tipos de entrada do Solver. O primeiro exemplo envolve um problema max-cut definido em um grafo 3-regular com 156 nós, enquanto o segundo aborda um problema de Cobertura Mínima de Vértices com 50 nós definido por uma função de custo.

Para obter acesso ao Optimization Solver, [entre em contato com o Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Descrição da função
O Solver otimiza e automatiza completamente todo o algoritmo, desde a supressão de erros no nível de hardware até o mapeamento eficiente de problemas e a otimização clássica em malha fechada. Por baixo dos panos, o pipeline do Solver reduz erros em cada etapa, possibilitando o desempenho aprimorado necessário para escalar de forma significativa. O fluxo de trabalho subjacente é inspirado no Quantum Approximate Optimization Algorithm (QAOA), que é um algoritmo híbrido quântico-clássico. Para um resumo detalhado do fluxo de trabalho completo do Optimization Solver, consulte [o manuscrito publicado](https://arxiv.org/abs/2406.01743).

![Visualização do fluxo de trabalho do Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Para resolver um problema genérico com o Optimization Solver:
1. Defina seu problema como uma função objetivo, um grafo ou uma cadeia de spin `SparsePauliOp`.
2. Conecte-se à função por meio do Catálogo de Qiskit Functions.
3. Execute o problema com o Solver e recupere os resultados.
### Formatos de problema aceitos
- Representação por expressão polinomial de uma função objetivo. Idealmente criada em Python com um objeto SymPy Poly existente e formatada como string usando [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Representação em grafo de um tipo específico de problema. O grafo deve ser criado usando a biblioteca networkx em Python e, em seguida, convertido para string usando a função networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Representação em cadeia de spin de um problema específico. A cadeia de spin deve ser representada como um objeto `SparsePauliOp`; consulte a [documentação](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) para mais detalhes.

> **Note:** Se você quiser usar um backend que esta função não suporta atualmente, [entre em contato com o Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) para adicionar suporte.
## Benchmarks
[Resultados de benchmarking publicados](https://arxiv.org/abs/2406.01743) mostram que o Solver resolve com sucesso problemas com mais de 120 qubits, superando até mesmo resultados previamente publicados em dispositivos de recozimento quântico e de íons aprisionados. As métricas de benchmark a seguir fornecem uma indicação aproximada da precisão e da escala dos tipos de problema com base em alguns exemplos. As métricas reais podem variar com base em diversas características do problema, como o número de termos na função objetivo (densidade) e sua localidade, o número de variáveis e a ordem polinomial.

O "Número de qubits" indicado não é uma limitação rígida, mas representa limites aproximados em que você pode esperar uma precisão de solução extremamente consistente. Problemas de maior escala já foram resolvidos com sucesso, e testes além desses limites são encorajados.

Conectividade arbitrária de qubits é suportada em todos os tipos de problema.

| Tipo de problema    | Número de qubits | Exemplo | Precisão | Tempo total (s) | Uso de runtime (s) | Número de iterações
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Problemas quadráticos com conexões esparsas  | 156 | max-cut 3-regular| 100%     | 1764     | 293          | 16 |
| Otimização binária de ordem superior | 156 | Modelo Ising spin-glass | 100%      | 1461     | 272          | 16 |
| Problemas quadráticos com conexões densas | 50 | max-cut totalmente conectado| 100%      |  1758    | 268  | 12 |
| Problema com restrições e termos de penalidade | 50 | Cobertura Mínima de Vértices Ponderada com 8% de densidade de arestas | 100%      | 1074     | 215 | 10 |
## Primeiros passos
Primeiro, autentique-se usando sua [chave de API do IBM Quantum](http://quantum.cloud.ibm.com/). Em seguida, selecione a Qiskit Function da seguinte forma. (Este trecho assume que você já [salvou sua conta](/guides/functions#install-qiskit-functions-catalog-client) no seu ambiente local.)

In [4]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

### 1. Definir o problema
Você pode executar um problema Max-Cut definindo um problema em grafo e especificando `problem_type='maxcut'`.

In [3]:
# %pip install networkx numpy

### 1. Define the problem
You can run a max-cut problem by defining a graph problem and specifying `problem_type='maxcut'`.

In [1]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [2]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

### 2. Executar o problema
Ao usar o método de entrada baseado em grafo, especifique o tipo de problema.

In [3]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Run the problem
When using the graph-based input method, specify the problem type.

In [4]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [ ]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions-get-started#check-job-status) or return [results](/docs/guides/functions-get-started#retrieve-results) as follows:

In [9]:
# Print the ID so you can use it later, if necessary
print(maxcut_job.job_id)

# Get job status
print(maxcut_job.status())

34b53970-d95a-4e24-8763-fc6f3d112843


QUEUED


### 3. Recuperar o resultado
Recupere o valor de corte ótimo do dicionário de resultados.

> **Note:** O mapeamento das variáveis para a bitstring pode ter mudado. O dicionário de saída contém um subdicionário `variables_to_bitstring_index_map`, que ajuda a verificar a ordenação.

In [10]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior max-cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can also solve constrained optimization problems by passing hard constraints directly to the Solver through the `constraint` input, instead of encoding them as penalty terms in the objective function. The Solver currently supports Hamming-weight-1 constraints: each constraint specifies a group of variables where exactly one variable must equal 1 and the rest must equal 0.

The following example demonstrates how to construct a cost function and a set of hard constraints for a constrained optimization problem, [graph partitioning](https://en.wikipedia.org/wiki/Graph_partition), by assigning every node in a graph to exactly one of several groups while minimizing the total weight of edges whose endpoints land in the same group.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [11]:
# %pip install numpy networkx sympy

Você pode verificar a precisão do resultado resolvendo o problema classicamente com solvers de código aberto como o [PuLP](https://coin-or.github.io/pulp/), caso o grafo não seja densamente conectado. Problemas de alta densidade podem exigir solvers clássicos avançados para validar a solução.
## Exemplo: Otimização com restrições
O exemplo anterior de max-cut é um problema comum de otimização binária quadrática sem restrições. O Optimization Solver do Q-CTRL pode ser usado para diversos tipos de problemas, incluindo otimização com restrições. Você pode resolver tipos de problema arbitrários fornecendo a definição do problema representada como um polinômio em que as restrições são modeladas como termos de penalidade.

O exemplo a seguir demonstra como construir uma função de custo para um problema de otimização com restrições, a [cobertura mínima de vértices](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
Além dos pacotes `qiskit-ibm-catalog` e `qiskit`, você também usará os seguintes pacotes para executar este exemplo: `numpy`, `networkx` e `sympy`. Você pode instalar esses pacotes descomentando a célula a seguir se estiver executando este exemplo em um notebook com o kernel IPython.

In [26]:
import networkx as nx
from sympy import Symbol, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
min_weight = -1.0
max_weight = 1.0
for i in graph.nodes:
    weight = (max_weight - min_weight) * _rng.random() + min_weight
    graph.add_node(i, weight=weight)

# Optionally, visualize the graph
nx.draw_networkx(graph, nx.kamada_kawai_layout(graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

### 1. Definir o problema
Defina um problema MVC aleatório gerando um grafo com nós ponderados aleatoriamente.

In [27]:
# Construct the cost function.
group_count = 3
variables = [
    Symbol(f"n[{i},{g}]")
    for i in range(node_count)
    for g in range(group_count)
]
node_group_var = {
    (i, g): variables[i * group_count + g]
    for i in range(node_count)
    for g in range(group_count)
}
cost_function = Poly(0, *variables)

for i, j in graph.edges():
    edge_weight = graph.nodes[i]["weight"] + graph.nodes[j]["weight"]
    for g in range(group_count):
        cost_function += (
            edge_weight * node_group_var[(i, g)] * node_group_var[(j, g)]
        )

![Saída da célula de código anterior](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

Um modelo de otimização padrão para MVC ponderado pode ser formulado da seguinte forma. Primeiro, uma penalidade deve ser adicionada para qualquer caso em que uma aresta não esteja conectada a um vértice no subconjunto. Portanto, seja $n_i = 1$ se o vértice $i$ está na cobertura (ou seja, no subconjunto) e $n_i = 0$ caso contrário. Em segundo lugar, o objetivo é minimizar o número total de vértices no subconjunto, o que pode ser representado pela seguinte função:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [28]:
# Build the hard constraint: exactly one group per node.
constraint_dict = {
    str(tuple(f"n[{i},{g}]" for g in range(group_count))): 1
    for i in range(node_count)
}
print(f"Problem constraints: {constraint_dict}")

Problem constraints: {"('n[0,0]', 'n[0,1]', 'n[0,2]')": 1, "('n[1,0]', 'n[1,1]', 'n[1,2]')": 1, "('n[2,0]', 'n[2,1]', 'n[2,2]')": 1, "('n[3,0]', 'n[3,1]', 'n[3,2]')": 1, "('n[4,0]', 'n[4,1]', 'n[4,2]')": 1, "('n[5,0]', 'n[5,1]', 'n[5,2]')": 1, "('n[6,0]', 'n[6,1]', 'n[6,2]')": 1, "('n[7,0]', 'n[7,1]', 'n[7,2]')": 1, "('n[8,0]', 'n[8,1]', 'n[8,2]')": 1, "('n[9,0]', 'n[9,1]', 'n[9,2]')": 1, "('n[10,0]', 'n[10,1]', 'n[10,2]')": 1, "('n[11,0]', 'n[11,1]', 'n[11,2]')": 1, "('n[12,0]', 'n[12,1]', 'n[12,2]')": 1, "('n[13,0]', 'n[13,1]', 'n[13,2]')": 1, "('n[14,0]', 'n[14,1]', 'n[14,2]')": 1, "('n[15,0]', 'n[15,1]', 'n[15,2]')": 1, "('n[16,0]', 'n[16,1]', 'n[16,2]')": 1, "('n[17,0]', 'n[17,1]', 'n[17,2]')": 1, "('n[18,0]', 'n[18,1]', 'n[18,2]')": 1, "('n[19,0]', 'n[19,1]', 'n[19,2]')": 1, "('n[20,0]', 'n[20,1]', 'n[20,2]')": 1, "('n[21,0]', 'n[21,1]', 'n[21,2]')": 1, "('n[22,0]', 'n[22,1]', 'n[22,2]')": 1, "('n[23,0]', 'n[23,1]', 'n[23,2]')": 1, "('n[24,0]', 'n[24,1]', 'n[24,2]')": 1, "('n[25,

Agora, cada aresta no grafo deve incluir pelo menos um ponto de extremidade da cobertura, o que pode ser expresso como a inequação:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Todo caso em que uma aresta não está conectada ao vértice de cobertura deve ser penalizado. Isso pode ser representado na função de custo adicionando uma penalidade da forma $P(1-n_i-n_j+n_i n_j)$, em que $P$ é uma constante de penalidade positiva. Assim, uma alternativa sem restrições para a inequação restrita para MVC ponderado é:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [20]:
# Solve the problem
partition_job = solver.run(
    problem=srepr(cost_function),
    constraint=constraint_dict,
    backend_name="ibm_marrakesh",  # E.g. "ibm_marrakesh"
)

### 2. Executar o problema

In [21]:
# Print the ID so you can use it later, if necessary
print(partition_job.job_id)

# Get job status
print(partition_job.status())

b8085944-f313-444e-be39-ea61b1b47ebd
QUEUED


Verifique o [status](/guides/functions#check-job-status) da carga de trabalho da sua Qiskit Function ou recupere os [resultados](/guides/functions#retrieve-results) da seguinte forma:

In [ ]:
partition_result = partition_job.result()
qctrl_cost = partition_result["solution_bitstring_cost"]
solution_bitstring = partition_result["solution_bitstring"]

# Print results
print(f"Total weight of same-group edges: {qctrl_cost}")
print(f"Solution bitstring: {solution_bitstring}")

Total weight of same-group edges: -36.5539
Solution bitstring: 100100100100100001100100100100100100100100100100100001010100010100100100100010001001100100100001100001100001010001001010100100100100100010100100100100


## Get support

For any questions or issues, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com).

## Changelog

- 2026-08-10: Added support for hard (Hamming weight 1) constraints via the `constraint` input, and updated the constrained optimization example to use them.
- 2026-02-11: We now have support for `ibm_miami`

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Q-CTRL Optimization Solver](https://quantum.cloud.ibm.com/functions?id=q-ctrl-optimization-solver).
- Visit the [API reference](/docs/api/functions/q-ctrl-optimization-solver) for this Qiskit Function.
- Try the [Solve higher-order binary optimization problems with Q-CTRL's Optimization Solver](/docs/tutorials/solve-higher-order-binary-optimization-problems-with-q-ctrls-optimization-solver) tutorial.
- Review [Sachdeva, N., et al. (2024).  Quantum optimization using a 127-qubit gate-model IBM quantum computer can outperform quantum annealers for nontrivial binary optimization problems. arXiv preprint arXiv:2406.01743](https://arxiv.org/abs/2406.01743).
- Review [Loco, D., et al. (2026).  Practical protein-pocket hydration-site prediction for drug discovery on a quantum computer. arXiv preprint arXiv:2512.08390](https://arxiv.org/abs/2512.08390).
- Review the [Mazda](https://q-ctrl.com/case-study/tackling-a-costly-bottleneck-in-automotive-design) case study.
- Review the [Network Rail](https://q-ctrl.com/case-study/accelerating-the-schedule-for-quantum-enhanced-rail) case study.
- Review the [Australian Army](https://q-ctrl.com/case-study/improving-army-logistics-with-quantum-computing) case study.
- Review the [Transport for New South Wales](https://q-ctrl.com/case-study/delivering-quantum-computing-for-faster-commuting) case study.

</Admonition>